In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("inventory.db")

df = pd.read_sql("SELECT * FROM vendor_invoice", conn)

df.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


In [4]:
print(df.columns)

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')


In [5]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "")

print(df.columns)

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')


In [6]:
# Feature Engineering (CORRECTED)

df["cost_per_unit"] = df["Freight"] / df["Quantity"]

df["vendor_avg_freight"] = df.groupby("VendorName")["Freight"].transform("mean")

df["freight_ratio"] = df["Freight"] / df["Dollars"]   # 👈 FIXED

df["deviation"] = df["Freight"] - df["vendor_avg_freight"]

df.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval,cost_per_unit,vendor_avg_freight,freight_ratio,deviation
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None,0.578333,1.559750,0.016195,1.910250
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None,0.571333,14.434727,0.060975,-5.864727
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None,0.922000,3.995094,0.043246,0.614906
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None,0.290614,1623.386727,0.021349,1311.813273
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None,0.221809,154.734727,0.027642,274.465273


In [7]:
X = df[[
    "Quantity",
    "Dollars",                # 👈 FIXED
    "vendor_avg_freight",
    "cost_per_unit",
    "freight_ratio",
    "deviation"
]]

y = df["Freight"]

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# model
model = RandomForestRegressor()

# train
model.fit(X_train, y_train)

print("✅ Model trained")

✅ Model trained


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestRegressor()

model.fit(X_train, y_train)

print("✅ Model trained")

✅ Model trained


In [11]:
import joblib

joblib.dump(model, "freight_model.pkl")

print("✅ Model saved")

✅ Model saved


In [12]:
from sklearn.ensemble import IsolationForest
import joblib

iso = IsolationForest(contamination=0.05)

iso.fit(X)

joblib.dump(iso, "fraud_model.pkl")

print("✅ Fraud model saved")

✅ Fraud model saved
